### **Drop**
Eliminar tablas logicas y rutas
- Por defecto **no borra nada**: con `confirmar = no` solo muestra lo que borraría (simulacro).
- Para borrar de verdad, cambiar el valor del widget `confirmar` a **`si`** y ejecutar nuevamente.
- El widget borrarCheckpoint = si, borra el checkpoint del auto loader de bronze_viajes_data del catalogo indicado, si se borran las tablas y no el checkpoint, bronze viajes cree que ya cargo los archivos y vuelve a cargar 0 filas
- El widget borrarClimaRaw = si, borra las carpetas que descargo bronze_clima 
- Después de un drop completo, el orden para reconstruir es: 0. Preparacion_Ambiente, 1. Bronze_clima, 1. Bronze_estaciones_data, 1. Bronze_viajes_data, 2. Silver_clima, 2. Silver_estaciones, 2. Silver_viajes, 3. Golden_modelo, 3. Golden_agregados, 4. Export Azure SQL

In [0]:
%python
dbutils.widgets.removeAll()

In [0]:
%python
dbutils.widgets.text("catalogo", "catalog_ecobici_dev")
dbutils.widgets.text("nameStorage", "adlsmartdataproy26")
dbutils.widgets.text("rutasExtra", "")
dbutils.widgets.dropdown("borrarCheckpoint", "si", ["si", "no"])
dbutils.widgets.dropdown("borrarClimaRaw", "no", ["si", "no"])
dbutils.widgets.dropdown("confirmar", "no", ["si", "no"])


In [0]:
%python
CATALOGO        = dbutils.widgets.get("catalogo").strip()
STORAGE         = dbutils.widgets.get("nameStorage").strip()
BORRAR_CLIMA    = dbutils.widgets.get("borrarClimaRaw") == "si"
BORRAR_CHECKPOINT = dbutils.widgets.get("borrarCheckpoint") == "si"
CONFIRMADO      = dbutils.widgets.get("confirmar") == "si"
RUTAS_EXTRA     = [r.strip() for r in dbutils.widgets.get("rutasExtra").split(",") if r.strip()]

In [0]:
%python
CAPAS = ["golden", "silver", "bronze"]          # se borran de la última a la primera
RUTA_CLIMA_RAW = f"abfss://raw@{STORAGE}.dfs.core.windows.net/datasets/clima"
# Checkpoint del Auto Loader de Bronze viajes: hay uno por catálogo (igual que en el notebook Bronze)
RUTA_CHECKPOINT = f"abfss://raw@{STORAGE}.dfs.core.windows.net/datasets/_checkpoints/{CATALOGO}/viajes_raw"

In [0]:

%python
## No borra los archivos de origen
PROTEGIDAS = ["/datasets/viajes", "/datasets/estaciones"]
for ruta in RUTAS_EXTRA:
    assert not any(ruta.rstrip("/").endswith(p) or f"{p}/" in ruta for p in PROTEGIDAS), \
        f"{ruta} apunta a los archivos de origen y no se puede borrar desde este notebook"

print("Catálogo :", CATALOGO)
print("Modo     :", "BORRAR" if CONFIRMADO else "SIMULACRO (no se borra nada)")

Catálogo : catalog_ecobici_dev
Modo     : SIMULACRO (no se borra nada)


Tablas existentes

In [0]:
%python
def nombre_esquema(capa):
    return f"{CATALOGO}.{capa}"


def nombre_tabla(capa, tabla):
    return f"{CATALOGO}.{capa}.`{tabla}`"


def objetos_de(capa):
    """Lista de (nombre, tipo) de lo que hay en el esquema. Si el esquema no existe, lista vacía."""
    try:
        return [(t.name, t.tableType) for t in spark.catalog.listTables(nombre_esquema(capa)) if not t.isTemporary]
    except Exception:
        return []


inventario = {capa: objetos_de(capa) for capa in CAPAS}
for capa in CAPAS:
    print(f"--- {capa}: {len(inventario[capa])} objetos ---")
    for nombre, tipo in inventario[capa]:
        print(f"   {nombre}  ({tipo})")

--- golden: 11 objetos ---
   agg_demanda_hora_dia  (MANAGED)
   agg_estaciones  (MANAGED)
   agg_resumen_diario  (MANAGED)
   agg_rutas_frecuentes  (MANAGED)
   agg_uso_por_clima  (MANAGED)
   dim_clima  (MANAGED)
   dim_estacion  (MANAGED)
   dim_fecha  (MANAGED)
   dim_usuario  (MANAGED)
   fact_clima_hora  (MANAGED)
   fact_viajes  (MANAGED)
--- silver: 7 objetos ---
   clima_cuarentena  (MANAGED)
   clima_hora  (MANAGED)
   estaciones  (MANAGED)
   estaciones_cuarentena  (MANAGED)
   resumen_calidad_viajes  (MANAGED)
   viajes  (MANAGED)
   viajes_cuarentena  (MANAGED)
--- bronze: 3 objetos ---
   clima_raw  (MANAGED)
   estaciones_raw  (MANAGED)
   viajes_raw  (MANAGED)


Eliminar tablas existentes

In [0]:
%python
for capa in CAPAS:
    for nombre, tipo in inventario[capa]:
        instruccion = f"DROP {'VIEW' if tipo == 'VIEW' else 'TABLE'} IF EXISTS {nombre_tabla(capa, nombre)}"
        if CONFIRMADO:
            spark.sql(instruccion)
            print("BORRADA  ", instruccion)
        else:
            print("SIMULACRO", instruccion)

SIMULACRO DROP TABLE IF EXISTS catalog_ecobici_dev.golden.`agg_demanda_hora_dia`
SIMULACRO DROP TABLE IF EXISTS catalog_ecobici_dev.golden.`agg_estaciones`
SIMULACRO DROP TABLE IF EXISTS catalog_ecobici_dev.golden.`agg_resumen_diario`
SIMULACRO DROP TABLE IF EXISTS catalog_ecobici_dev.golden.`agg_rutas_frecuentes`
SIMULACRO DROP TABLE IF EXISTS catalog_ecobici_dev.golden.`agg_uso_por_clima`
SIMULACRO DROP TABLE IF EXISTS catalog_ecobici_dev.golden.`dim_clima`
SIMULACRO DROP TABLE IF EXISTS catalog_ecobici_dev.golden.`dim_estacion`
SIMULACRO DROP TABLE IF EXISTS catalog_ecobici_dev.golden.`dim_fecha`
SIMULACRO DROP TABLE IF EXISTS catalog_ecobici_dev.golden.`dim_usuario`
SIMULACRO DROP TABLE IF EXISTS catalog_ecobici_dev.golden.`fact_clima_hora`
SIMULACRO DROP TABLE IF EXISTS catalog_ecobici_dev.golden.`fact_viajes`
SIMULACRO DROP TABLE IF EXISTS catalog_ecobici_dev.silver.`clima_cuarentena`
SIMULACRO DROP TABLE IF EXISTS catalog_ecobici_dev.silver.`clima_hora`
SIMULACRO DROP TABLE IF E

Eliminar rutas (checkpoints y clima descargado)

In [0]:
%python
rutas = list(RUTAS_EXTRA)
if BORRAR_CHECKPOINT:
    rutas.append(RUTA_CHECKPOINT)
if BORRAR_CLIMA:
    rutas.append(RUTA_CLIMA_RAW)

if not rutas:
    print("No hay rutas por borrar (borrarCheckpoint = no, borrarClimaRaw = no y rutasExtra vacío).")

for ruta in rutas:
    if not CONFIRMADO:
        print("SIMULACRO  borraría", ruta)
        continue
    try:
        resultado = dbutils.fs.rm(ruta, True)
        print("BORRADA   " if resultado else "NO EXISTÍA ", ruta)
    except Exception as error:
        print("NO SE PUDO BORRAR", ruta, "->", str(error)[:200])

SIMULACRO  borraría abfss://raw@adlsmartdataproy26.dfs.core.windows.net/datasets/_checkpoints/viajes_raw


Verificación 

In [0]:
%python
for capa in CAPAS:
    restantes = objetos_de(capa)
    print(f"{capa:8s} quedan {len(restantes)} objetos")
    if CONFIRMADO:
        assert not restantes, f"En {capa} todavía quedan: {[n for n, _ in restantes]}"
print("Listo." if CONFIRMADO else "Simulacro terminado: cambia 'confirmar' a 'si' para borrar.")

golden   quedan 11 objetos
silver   quedan 7 objetos
bronze   quedan 3 objetos
Simulacro terminado: cambia 'confirmar' a 'si' para borrar.
